In [1]:
import os
from pathlib import Path
from src.utils import load_json
from src.chem_draw import draw_reaction
from IPython.display import SVG
import pandas as pd
import polars as pl
from collections import defaultdict
import ipywidgets as widgets
from src.config import filepaths
%load_ext memory_profiler
print(pd.__version__)
print(pl.__version__)

2.2.3
1.20.0


In [2]:
assets_dir = Path(os.environ.get("BOTTLE_EXPANSION_ASSETS"))

In [3]:
assets = {}
for p in assets_dir.glob('*'):
    if p.suffix == '.json':
        assets[p.stem] = load_json(p)

    else:
        assets['svgs'] = {}
        for s in p.glob('*'):
            assets['svgs'][s.stem] = SVG(s).data

In [ ]:
tmp = defaultdict(list)
for k, v in assets['found_paths'].items():
    for _k, _v in v.items():    
        tmp[_k].append(_v)

df = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

df.to_parquet(
    filepaths['assets_explo'] / 'paths.parquet',
    index=False,
)

df.head()

In [ ]:
len(df)

In [ ]:
tmp = defaultdict(list)
for k, v in assets['known_reactions'].items():
    for _k, _v in v.items():

        if _k == 'image':
            continue
        
        tmp[_k].append(_v)

df = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

df.to_parquet(
    filepaths['assets_explo'] / 'known_reactions_no_image.parquet',
    index=False,
)

df.head()

In [ ]:
tmp = defaultdict(list)
for k, v in assets['known_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        if _k == 'image':
            tmp[_k].append(assets['svgs'][_v.removesuffix('.svg')])
        else:
            tmp[_k].append(_v)

krs = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

krs.to_parquet(
    filepaths['assets_explo'] / 'known_reactions.parquet',
    index=False,
)

krs.head()

In [ ]:
tmp = defaultdict(list)
for k, v in assets['predicted_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        if _k == 'image':
            tmp[_k].append(assets['svgs'][_v.removesuffix('.svg')])
        else:
            tmp[_k].append(_v)

prs = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

all_rxns = pd.concat([krs, prs], ignore_index=True)

all_rxns.to_parquet(
    filepaths['assets_explo'] / 'reactions.parquet',
    index=False,
)

all_rxns.head()

In [ ]:
tmp = defaultdict(list)
to_enz = defaultdict(list)
for k, v in assets['known_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        if _k == 'image':
            tmp[_k].append(assets['svgs'][_v.removesuffix('.svg')])
        elif _k == 'enzymes':
            tmp[_k].append([e['uniprot_id'] for e in _v])

            for e in _v:
                for ek, ev in e.items():
                    to_enz[ek].append(ev)

        else:
            tmp[_k].append(_v)

krs = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

all_rxns = pd.concat([krs, prs], ignore_index=True)

all_rxns.to_parquet(
    filepaths['assets_explo'] / 'reactions_no_enzymes.parquet',
    index=False,
)

enzymes = pd.DataFrame(
    data=to_enz,
    columns=to_enz.keys()
)

enzymes.to_parquet(
    filepaths['assets_explo'] / 'enzymes.parquet',
    index=False,
)

enzymes.head()

In [ ]:
to_enz

In [ ]:
tmp = list(assets['svgs'].items())

df = pd.DataFrame(
    data=tmp,
    columns=['reaction_id', 'image']
)

df.to_parquet(
    filepaths['assets_explo'] / 'svgs.parquet',
    index=False,
)


df.head()

json/dict, pandas, polars head-to-head

In [21]:
# Create pandas and polars dataframes for reactions

krs_dict = defaultdict(list)
for k, v in assets['known_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        krs_dict[_k].append(_v)

prs_dict = defaultdict(list)
for k, v in assets['predicted_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        prs_dict[_k].append(_v)

rxns_pd = pd.concat(
    (pd.DataFrame(data=krs_dict), pd.DataFrame(data=prs_dict)),
    ignore_index=True
)

print(rxns_pd.memory_usage(deep=True).sum() / 1e6)
print(rxns_pd.shape)
rxns_pd.head()


# Save each to parquet
# all_rxns.to_parquet(
#     filepaths['assets_explo'] / 'reactions.parquet',
#     index=False,
# )

10.29877
(9034, 9)


,id,smarts,operators,enzymes,db_entries,reaction_center,image,analogues,rcmcs
0,6615,CCC(C)C1C2(C)C(=O)C(=CO)C(O)(C3CC(C)(O)CC(C)C3...,"[rule0004, rule0004_13]","[{'uniprot_id': 'Q0UK51', 'sequence': 'MAAYFLL...","[{'name': 'rhea', 'id': 61905}]","[[3], [3, 4, 5, 45, 46, 47], [0, 1]]",6615.svg,NaN,NaN
1,7757,CC(C)=CCNc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1O...,"[rule0004, rule0004_13]","[{'uniprot_id': 'Q9ZW95', 'sequence': 'MMVTLVL...","[{'name': 'rhea', 'id': 47813}]","[[0], [3, 4, 5, 45, 46, 47], [0, 1]]",7757.svg,NaN,NaN
2,8249,CCC1(O)Cc2c(O)c3c(c(O)c2C(OC2CC(N)C(O)C(C)O2)C...,"[rule0004, rule0004_13]","[{'uniprot_id': 'Q9ZAU3', 'sequence': 'MAVDPFA...","[{'name': 'rhea', 'id': 56361}]","[[1], [3, 4, 5, 45, 46, 47], [0, 1]]",8249.svg,NaN,NaN
3,8882,CC(C=O)CCCC(C)C1CCC2C3CC=C4CC(O)CCC4(C)C3CCC12...,"[rule0004, rule0004_13]","[{'uniprot_id': 'P9WPL5', 'sequence': 'MTEAPDV...","[{'name': 'rhea', 'id': 43845}]","[[2], [3, 4, 5, 45, 46, 47], [0, 1]]",8882.svg,NaN,NaN
4,10657,CC(C)=CCNc1ncnc2c1ncn2C1OC(COP(=O)(O)OP(=O)(O)...,"[rule0004, rule0004_13]","[{'uniprot_id': 'Q9ZW95', 'sequence': 'MMVTLVL...","[{'name': 'rhea', 'id': 47817}]","[[0], [3, 4, 5, 45, 46, 47], [0, 1]]",10657.svg,NaN,NaN


In [25]:
rxns_pl = pl.concat(
    (pl.DataFrame(krs_dict), pl.DataFrame(prs_dict)),
    how='diagonal'
)

print(rxns_pl.estimated_size(unit='mb'))
print(len(rxns_pl))
rxns_pl.head()

163.24068927764893
9034


id,smarts,operators,enzymes,db_entries,reaction_center,image,analogues,rcmcs
str,str,list[str],list[struct[7]],list[struct[2]],list[list[i64]],str,list[str],struct[20]
"""6615""","""CCC(C)C1C2(C)C(=O)C(=CO)C(O)(C…","[""rule0004"", ""rule0004_13""]","[{""Q0UK51"",""MAAYFLLGLYGSTLVYRLIFHPLNRFPGPLAARISDLWLCTQLGGHDMHHLSERLSKRYGEFVRIGSSTLMLTHPKAVAAIYGPGSPCRKGTFYDLEQPNRGIATRDESLHAGRRRVWSRGFGDKALRTYEPRVAAYVHMLLGRLADARGKPVDMARLAEAFAFDTMGDLGLGADFGMLRQARTHEAVEQLVQGMTIMGRRLPMWLMRLLIDVAQALVPTAATTGFLGFCHHHLDRFMADPRRSERPSLMAPLLSHYEKQNIADRDLSILRNDCRFIIIAGSDTVAATLTFAFFYLAKHPGHVTRLREELFPLRAADGTFSHQRIFDAPHLNAVINETLRLHPPASTIPRVTPPQGLVVADTFIPGDMTVFSSQYALGRSEAIYSKASDFIPERWCSRPDLIKDGSAYAPFSIGHHSCLGRPLALMEMRLVLAETLSRFDIAFAPGFDANHFLQHVHDCMSWHIGKLGLTFTAIE"",""Evidence at protein level"",""reviewed"",""1.-.-.-"",""Phaeosphaeria nodorum (strain SN15 / ATCC MYA-4574 / FGSC 10173) (Glume blotch fungus) (Parastagonospora nodorum)"",""Cytochrome P450 monooxygenase sthD (EC 1.-.-.-) (Stemphyloxin II biosynthesis cluster protein D)""}]","[{""rhea"",61905}]","[[3], [3, 4, … 47], [0, 1]]","""6615.svg""",null,null
"""7757""","""CC(C)=CCNc1ncnc2c1ncn2C1OC(COP…","[""rule0004"", ""rule0004_13""]","[{""Q9ZW95"",""MMVTLVLKYVLVIVMTLILRVLYDSICCYFLTPRRIKKFMERQGITGPKPRLLTGNIIDISKMLSHSASNDCSSIHHNIVPRLLPHYVSWSKQYGKRFIMWNGTEPRLCLTETEMIKELLTKHNPVTGKSWLQQQGTKGFIGRGLLMANGEAWHHQRHMAAPAFTRDRLKGYAKHMVECTKMMAERLRKEVGEEVEIGEEMRRLTADIISRTEFGSSCDKGKELFSLLTVLQRLCAQATRHLCFPGSRFLPSKYNREIKSLKTEVERLLMEIIDSRKDSVEIGRSSSYGDDLLGLLLNQMDSNKNNLNVQMIMDECKTFFFTGHETTSLLLTWTLMLLAHNPTWQDNVRDEVRQVCGQDGVPSVEQLSSLTSLNKVINESLRLYPPATLLPRMAFEDIKLGDLIIPKGLSIWIPVLAIHHSNELWGEDANEFNPERFTTRSFASSRHFMPFAAGPRNCIGQTFAMMEAKIILAMLVSKFSFAISENYRHAPIVVLTIKPKYGVQLVLKPLDL"",""Evidence at protein level"",""reviewed"",""1.14.13.-"",""Arabidopsis thaliana (Mouse-ear cress)"",""Cytokinin hydroxylase (EC 1.14.13.-) (Cytochrome P450 35A2)""}, {""Q9FF18"",""MLLTILKSLLVIFVTTILRVLYDTISCYWLTPRRIKKIMEQQGVTGPKPRPLTGNILEISAMVSQSASKDCDSIHHDIVGRLLPHYVAWSKQYGKRFIVWNGTDPRLCLTETELIKELLMKHNGVSGRSWLQQQGTKNFIGRGLLMANGQDWHHQRHLAAPAFTGERLKGYARHMVECTSKLVERLRKEVGEGANEVEIGEEMHKLTADIISRTKFGSSFEKGKELFNHLTVLQRRCAQATRHLCFPGSRFLPSKYNREIKSLKKEVERLLIEIIQSRRDCAEMGRSSTHGDDLLGLLLNEMDIDKNNNNNNNNLQLIMDECKTFFFAGHETTALLLTWTTMLLADNPTWQEKVREEVREVFGRNGLPSVDQLSKLTSLSKVINESLRLYPPATLLPRMAFEDLKLGDLTIPKGLSIWIPVLAIHHSEELWGKDANQFNPERFGGRPFASGRHFIPFAAGPRNCIGQQFALMEAKIILATLISKFNFTISKNYRHAPIVVLTIKPKYGVQVILKPLVS"",""Evidence at protein level"",""reviewed"",""1.14.13.-"",""Arabidopsis thaliana (Mouse-ear cress)"",""Cytokinin hydroxylase (EC 1.14.13.-) (Cytochrome P450 35A1)""}]","[{""rhea"",47813}]","[[0], [3, 4, … 47], [0, 1]]","""7757.svg""",null,null
"""8249""","""CCC1(O)Cc2c(O)c3c(c(O)c2C(OC2C…","[""rule0004"", ""rule0004_13""]","[{""Q9ZAU3"",""MAVDPFACPMMTMQRKPEVHDAFREAGPVVEVNAPAGGPAWVITDDALAREVLADPRFVKDPDLAPAAWRGVDDGLDIPVPELRPFTLIAVDGEAHRRLRRIHAPAFNPRRLAERTDRIAAIAGRLLTELADASGRSGKPAELIGGFAYHFPLLVICELLGVPVTDPAMAREAVSVLKALGLGGPQSGGGDGTDPAGGVPDTSALESLLLEAVHSARRNDTPTMTRVLYERAQAEFGSVSDDQLVYMITGLIFAGHDTTGSFLGFLLAEVLAGRLAADADEDAVSRFVEEALRYHPPVPYTLWRFAATEVTIGGVRLPRGAPVLVDIEGTNTDGRHHDAPHAFHPDRPSWRRLTFGDGPHYCIGEQLAQLESRTMIGVLRSRFPEARLAVPYDELRWCRKGAQTARLTELPVWLR"",""Evidence at protein level"",""reviewed"",""1.14.13.181"",""Streptomyces peucetius"",""Cytochrome P-450 monooxygenase DoxA (EC 1.14.13.181) (13-deoxycarminomycin C-13 hydroxylase) (13-deoxydaunorubicin C-13 hydroxylase) (13-dihydrocarminomycin C-13 hydroxylase) (13-dihydrodaunorubicin C-13 hydroxylase)""}, {""Q93MI2"",""MSGEAPRVAVDPFACPMMTMQRKPEVHDAFREAGPVVEVNAPAGGPAWFITDDALSRYVLADPRLVKDPDLAPAAWRGVVDGLDIPVPELRPFTLIAVDGEAHRRLHRIHAPAFNPRRLAERTDRIAAIAGRLLTELADASGRSGEPAELIGGFAYHFPLLVICELLGVPVTVPMAREAVSVLKALASAAQSGGGDGTDPAGGVPDTSALESLLLEAVHSARRNDTPTMTRVLYEHTQAEFGSVSDNQLVYMITGIIFAGHERTGSFLGFLLAEVLAGRLAADADEDAVSRFVEEAVRYHPPVPYTLWRFAATEVTIGGVRLPPGAPVLVDIEGTNTDGRHHDAPHAFHPDRPSWRRLTFGDGPHYCIGEQLAQLESRTMIGVLRSRFPEARLAVPYDELRWCRNGAQTARLTELPVWLR"",""Evidence at protein level"",""reviewed"",""1.14.13.181"",""Streptomyces peucetius sub

In [26]:
rxns_pl_from_pd = pl.from_pandas(rxns_pd)
print(rxns_pl_from_pd.estimated_size(unit='mb'))
print(len(rxns_pl_from_pd))
rxns_pl_from_pd.head()

436.9331121444702
9034


id,smarts,operators,enzymes,db_entries,reaction_center,image,analogues,rcmcs
str,str,list[str],list[struct[7]],list[struct[2]],list[list[i64]],str,list[str],struct[5045]
"""6615""","""CCC(C)C1C2(C)C(=O)C(=CO)C(O)(C…","[""rule0004"", ""rule0004_13""]","[{""1.-.-.-"",""Evidence at protein level"",""Cytochrome P450 monooxygenase sthD (EC 1.-.-.-) (Stemphyloxin II biosynthesis cluster protein D)"",""Phaeosphaeria nodorum (strain SN15 / ATCC MYA-4574 / FGSC 10173) (Glume blotch fungus) (Parastagonospora nodorum)"",""reviewed"",""MAAYFLLGLYGSTLVYRLIFHPLNRFPGPLAARISDLWLCTQLGGHDMHHLSERLSKRYGEFVRIGSSTLMLTHPKAVAAIYGPGSPCRKGTFYDLEQPNRGIATRDESLHAGRRRVWSRGFGDKALRTYEPRVAAYVHMLLGRLADARGKPVDMARLAEAFAFDTMGDLGLGADFGMLRQARTHEAVEQLVQGMTIMGRRLPMWLMRLLIDVAQALVPTAATTGFLGFCHHHLDRFMADPRRSERPSLMAPLLSHYEKQNIADRDLSILRNDCRFIIIAGSDTVAATLTFAFFYLAKHPGHVTRLREELFPLRAADGTFSHQRIFDAPHLNAVINETLRLHPPASTIPRVTPPQGLVVADTFIPGDMTVFSSQYALGRSEAIYSKASDFIPERWCSRPDLIKDGSAYAPFSIGHHSCLGRPLALMEMRLVLAETLSRFDIAFAPGFDANHFLQHVHDCMSWHIGKLGLTFTAIE"",""Q0UK51""}]","[{61905,""rhea""}]","[[3], [3, 4, … 47], [0, 1]]","""6615.svg""",null,null
"""7757""","""CC(C)=CCNc1ncnc2c1ncn2C1OC(COP…","[""rule0004"", ""rule0004_13""]","[{""1.14.13.-"",""Evidence at protein level"",""Cytokinin hydroxylase (EC 1.14.13.-) (Cytochrome P450 35A2)"",""Arabidopsis thaliana (Mouse-ear cress)"",""reviewed"",""MMVTLVLKYVLVIVMTLILRVLYDSICCYFLTPRRIKKFMERQGITGPKPRLLTGNIIDISKMLSHSASNDCSSIHHNIVPRLLPHYVSWSKQYGKRFIMWNGTEPRLCLTETEMIKELLTKHNPVTGKSWLQQQGTKGFIGRGLLMANGEAWHHQRHMAAPAFTRDRLKGYAKHMVECTKMMAERLRKEVGEEVEIGEEMRRLTADIISRTEFGSSCDKGKELFSLLTVLQRLCAQATRHLCFPGSRFLPSKYNREIKSLKTEVERLLMEIIDSRKDSVEIGRSSSYGDDLLGLLLNQMDSNKNNLNVQMIMDECKTFFFTGHETTSLLLTWTLMLLAHNPTWQDNVRDEVRQVCGQDGVPSVEQLSSLTSLNKVINESLRLYPPATLLPRMAFEDIKLGDLIIPKGLSIWIPVLAIHHSNELWGEDANEFNPERFTTRSFASSRHFMPFAAGPRNCIGQTFAMMEAKIILAMLVSKFSFAISENYRHAPIVVLTIKPKYGVQLVLKPLDL"",""Q9ZW95""}, {""1.14.13.-"",""Evidence at protein level"",""Cytokinin hydroxylase (EC 1.14.13.-) (Cytochrome P450 35A1)"",""Arabidopsis thaliana (Mouse-ear cress)"",""reviewed"",""MLLTILKSLLVIFVTTILRVLYDTISCYWLTPRRIKKIMEQQGVTGPKPRPLTGNILEISAMVSQSASKDCDSIHHDIVGRLLPHYVAWSKQYGKRFIVWNGTDPRLCLTETELIKELLMKHNGVSGRSWLQQQGTKNFIGRGLLMANGQDWHHQRHLAAPAFTGERLKGYARHMVECTSKLVERLRKEVGEGANEVEIGEEMHKLTADIISRTKFGSSFEKGKELFNHLTVLQRRCAQATRHLCFPGSRFLPSKYNREIKSLKKEVERLLIEIIQSRRDCAEMGRSSTHGDDLLGLLLNEMDIDKNNNNNNNNLQLIMDECKTFFFAGHETTALLLTWTTMLLADNPTWQEKVREEVREVFGRNGLPSVDQLSKLTSLSKVINESLRLYPPATLLPRMAFEDLKLGDLTIPKGLSIWIPVLAIHHSEELWGKDANQFNPERFGGRPFASGRHFIPFAAGPRNCIGQQFALMEAKIILATLISKFNFTISKNYRHAPIVVLTIKPKYGVQVILKPLVS"",""Q9FF18""}]","[{47813,""rhea""}]","[[0], [3, 4, … 47], [0, 1]]","""7757.svg""",null,null
"""8249""","""CCC1(O)Cc2c(O)c3c(c(O)c2C(OC2C…","[""rule0004"", ""rule0004_13""]","[{""1.14.13.181"",""Evidence at protein level"",""Cytochrome P-450 monooxygenase DoxA (EC 1.14.13.181) (13-deoxycarminomycin C-13 hydroxylase) (13-deoxydaunorubicin C-13 hydroxylase) (13-dihydrocarminomycin C-13 hydroxylase) (13-dihydrodaunorubicin C-13 hydroxylase)"",""Streptomyces peucetius"",""reviewed"",""MAVDPFACPMMTMQRKPEVHDAFREAGPVVEVNAPAGGPAWVITDDALAREVLADPRFVKDPDLAPAAWRGVDDGLDIPVPELRPFTLIAVDGEAHRRLRRIHAPAFNPRRLAERTDRIAAIAGRLLTELADASGRSGKPAELIGGFAYHFPLLVICELLGVPVTDPAMAREAVSVLKALGLGGPQSGGGDGTDPAGGVPDTSALESLLLEAVHSARRNDTPTMTRVLYERAQAEFGSVSDDQLVYMITGLIFAGHDTTGSFLGFLLAEVLAGRLAADADEDAVSRFVEEALRYHPPVPYTLWRFAATEVTIGGVRLPRGAPVLVDIEGTNTDGRHHDAPHAFHPDRPSWRRLTFGDGPHYCIGEQLAQLESRTMIGVLRSRFPEARLAVPYDELRWCRKGAQTARLTELPVWLR"",""Q9ZAU3""}, {""1.14.13.181"",""Evidence at protein level"",""Cytochrome P-450 monooxygenase DoxA (EC 1.14.13.181) (13-deoxycarminomycin C-13 hydroxylase) (13-deoxydaunorubicin C-13 hydroxylase) (13-dihydrocarminomycin C-13 hydroxylase) (13-dihydrodaunorubicin C-13 hydroxylase) (Daunorubicin C-14 hydroxylase)"",""Streptomyces peucetius subsp. caesius"",""reviewed"",""MSGEAPRVAVDPFACPMMTMQRKPEVHDAFREAGPVVEVNAPAGGPAWFITDDALSRYVLADPRLVKDPDLAPAAWRGVVDGLDIPVPELRPFTLIAVDGEAHRRLHRIHAPAFNPRRLAERTDRIAAIAGRLLTELADASGRSGEPAELIGGFAYHFPLLVICE

In [29]:
krs_pd = pd.DataFrame(krs_dict)
print(krs_pd.memory_usage(deep=True).sum() / 1e6)

krs_pl = pl.DataFrame(krs_dict)
print(krs_pl.estimated_size(unit='mb'))

4.302983
160.15295219421387


In [31]:
krs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5045 entries, 0 to 5044
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               5045 non-null   object
 1   smarts           5045 non-null   object
 2   operators        5045 non-null   object
 3   enzymes          5045 non-null   object
 4   db_entries       5045 non-null   object
 5   reaction_center  5045 non-null   object
 6   image            5045 non-null   object
dtypes: object(7)
memory usage: 276.0+ KB


In [37]:
krs_pd = pd.DataFrame(krs_dict).loc[:, ['enzymes']]
print(krs_pd.memory_usage(deep=True).sum() / 1e6)

krs_pl = pl.DataFrame(krs_dict)[:, ['enzymes']]
print(krs_pl.estimated_size(unit='mb'))

1.324284
158.6238431930542


In [39]:
type(krs_pd.loc[0, 'enzymes'])

list

In [43]:
type(krs_pl[0, 'enzymes'])

polars.series.series.Series

In [36]:
krs_pd = pd.DataFrame(krs_dict).loc[:, ['smarts', 'image']]
print(krs_pd.memory_usage(deep=True).sum() / 1e6)

krs_pl = pl.DataFrame(krs_dict)[:, ['smarts', 'image']]
print(krs_pl.estimated_size(unit='mb'))

1.525014
0.9827346801757812


In [44]:
krs_pd = pd.DataFrame(krs_dict).loc[:, ['reaction_center']]
print(krs_pd.memory_usage(deep=True).sum() / 1e6)

krs_pl = pl.DataFrame(krs_dict)[:, ['reaction_center']]
print(krs_pl.estimated_size(unit='mb'))

0.408868
0.28635406494140625


Loading in

In [17]:
%%memit
lrxns = pd.read_parquet(filepaths['assets_explo'] / 'reactions.parquet')

peak memory: 4916.50 MiB, increment: 3535.63 MiB


In [3]:
%%memit
lrxns = pd.read_parquet(filepaths['assets_explo'] / 'known_reactions_no_image.parquet')

peak memory: 526.80 MiB, increment: 290.42 MiB


In [3]:
lpaths = pd.read_parquet(filepaths['assets_explo'] / 'paths.parquet')

In [4]:
lpaths.head()

,id,starter,target,reactions,mdf,dG_opt,dG_err,sid,tid
0,Pe51c2d9fa49e794a7039aa32f6ac1c886acdf7eb,dmb,dmhb,[Re69b39c63c0593fa41a1cec058394da2e621d3af1d43...,445.840950,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,Cce065e1e7b273fd0df7f455452f948b30517b232,Cc2274a1b151121868789bd03eb37b9c4065ef3d1
1,Pe360ba4742c82490957272314621e9bb026f239d,pivalic_acid,3hpa,[R022d15344a76dbd79cf46e8c67679ce6aff193011ecd...,441.450972,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,C743d3d3a4a0e2cf0e31d4d7623c0d79884571a95,Ca44a84be6f833fe631009a55d05e4807de958fe0
2,P40ade8fddc6ffc08258f04c19c0b046bb0f774e0,pivalic_acid,3hpa,[R17b5116b7d8f16bf7d458ac6b368477df85a3f4374ad...,-601.396457,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,C743d3d3a4a0e2cf0e31d4d7623c0d79884571a95,Ca44a84be6f833fe631009a55d05e4807de958fe0
3,P46f3215e2dfc5800c477ad754a31024711fb488b,pivalic_acid,3hpa,[R17b5116b7d8f16bf7d458ac6b368477df85a3f4374ad...,-601.396459,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,C743d3d3a4a0e2cf0e31d4d7623c0d79884571a95,Ca44a84be6f833fe631009a55d05e4807de958fe0
4,P2121d91ef983b457832b344902d0474845935af0,pivalic_acid,3hpa,[R17b5116b7d8f16bf7d458ac6b368477df85a3f4374ad...,NaN,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,{'R001afef0a5241a44566b1a0b38ce9ec46d6c7cfc4b1...,C743d3d3a4a0e2cf0e31d4d7623c0d79884571a95,Ca44a84be6f833fe631009a55d05e4807de958fe0


Draw on the fly vs precompute images

In [ ]:
%%timeit -n 1_00
svgfig = draw_reaction(all_rxns.loc[0, 'smarts'], auto_scl=True)
svg = svgfig.to_str()

In [ ]:
%%timeit -n 1_00
svg = widgets.Image.from_file(assets_dir / 'svgs' / f"{all_rxns.loc[0, 'id']}.svg")

In [ ]:
%%timeit -n 1_00
svg = widgets.Image.from_file(assets_dir / 'svgs' / f"{all_rxns.loc[0, 'id']}.svg")

In [16]:
%%timeit -n 1_00
svg = lrxns.loc[0, 'image']
widgets.HTML(value=svg)

2.82 ms ± 630 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
